## Import necessary library

In [13]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import MinMaxScaler
from torch import nn
from torch.utils.data import DataLoader

## Get File path and load the csv data

In [14]:
from util.data_path import cassava_price_avg
from util.data_path import corn_price_avg
from util.data_path import green_bean_price_avg
from util.data_path import soybean_price_avg

In [15]:
# Function to process each CSV file
def process_csv(data_file):
    wide = pd.read_csv(data_file)
    long = wide.melt(id_vars="year", var_name="month", value_name="price").sort_values(
        ["year", "month"]
    )

    long["date"] = pd.to_datetime(
        (long["year"] - 543).astype(str)
        + "-"
        + long["month"].astype(str).str.zfill(2)
        + "-01"
    )
    long = long.set_index("date").sort_index()
    return long


# Process each crop data
cassava_df = process_csv(cassava_price_avg)
corn_df = process_csv(corn_price_avg)
green_bean_df = process_csv(green_bean_price_avg)
soybean_df = process_csv(soybean_price_avg)

# Add crop name column to each dataframe
cassava_df["crop"] = "cassava"
corn_df["crop"] = "corn"
green_bean_df["crop"] = "green_bean"
soybean_df["crop"] = "soybean"

# Create a combined dataframe for all crops
all_crops_df = pd.concat([cassava_df, corn_df, green_bean_df, soybean_df])

In [16]:
normalized_df = all_crops_df.copy()

yearly_avg = all_crops_df.groupby(["crop", "year"])["price"].mean().reset_index()
yearly_avg.rename(columns={"price": "yearly_avg_price"}, inplace=True)

normalized_df = pd.merge(normalized_df, yearly_avg, on=["crop", "year"])
normalized_df["normalized_price"] = (
    normalized_df["price"] / normalized_df["yearly_avg_price"]
)

# Create a more structured result dataframe
result_df = normalized_df[
    ["year", "month", "crop", "price", "yearly_avg_price", "normalized_price"]
]
result_df = result_df.sort_values(["crop", "year", "month"])